In [1]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 33.7 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,542 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu j

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from google.colab import drive

In [3]:
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Install ChromeDriver in Google Colab
def setup_selenium():
    !apt-get update > /dev/null 2>&1
    !apt install chromium-chromedriver > /dev/null 2>&1
    os.environ["PATH"] += ":/usr/lib/chromium-browser/"

setup_selenium()

In [5]:
def scrape_all_imdb_reviews(movie_id, movie_name, max_clicks=500):
    """Scrapes all IMDb reviews, ratings, and dates for a given movie ID using Selenium, and adds Movie Name."""
    url = f"https://www.imdb.com/title/{movie_id}/reviews/?ref_=tt_ururv_sm"
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--remote-debugging-port=9222")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(5)  # Allow page to load

    for _ in range(max_clicks):
      try:
          driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
          time.sleep(2)

          load_more_button = driver.find_element(By.XPATH, "//button[contains(@class, 'ipc-btn ipc-btn--single-padding ipc-btn--center-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button ipc-see-more__button')]")
          driver.execute_script("arguments[0].click();", load_more_button)
          time.sleep(2)
      except Exception as e:
          print(f"No more 'Load More' button found. {str(e)}")
          break

    # Click all 'Spoilers' buttons to reveal hidden reviews
    while True:
        try:
            spoiler_buttons = driver.find_elements(By.XPATH, "//button[contains(@class, 'ipc-btn ipc-btn--single-padding ipc-btn--center-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-error ipc-text-button sc-27c64f4e-1 joVDDm review-spoiler-button')]")
            if not spoiler_buttons:
                break
            for button in spoiler_buttons:
                driver.execute_script("arguments[0].click();", button)
                time.sleep(1)
        except Exception:
            print("No more spoilers buttons found.")
            break

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    reviews = []
    review_containers = soup.find_all("article", class_="sc-571af6d2-1 hzjHJm user-review-item")

    for review in review_containers:
        text = review.find("div", class_="ipc-html-content-inner-div")
        rating = review.find("span", class_="ipc-rating-star ipc-rating-star--base ipc-rating-star--otherUserAlt review-rating")
        date = review.find("div", class_="sc-a64e3cc9-1 bFvZOY")
        if date:
            date = date.find("li", class_="ipc-inline-list__item review-date")

        review_text = text.get_text(strip=True) if text else "No review text"
        if review_text == "No review text":
            hidden_text = review.find("span", class_="sc-16ede01-2 gkqnAC")
            review_text = hidden_text.get_text(strip=True) if hidden_text else "No review text"

        review_rating = rating.find("span").get_text(strip=True) if rating else "No rating"
        review_date = date.get_text(strip=True) if date else "No date"

        reviews.append({
            "Movie_ID": movie_id,
            "Movie_Name": movie_name,
            "Date": review_date,
            "Review": review_text,
            "Rating": review_rating,
        })

    print(f"Scraping completed. Total reviews collected: {len(reviews)}")
    df = pd.DataFrame(reviews)
    save_path = f"/content/drive/My Drive/Tugas Akhir/Dataset/{movie_id}_reviews.csv"
    df.to_csv(save_path, index=False)
    print(f"Reviews saved to {save_path}")
    return df

In [6]:
## Langsung ketik Movie_ID dan Movie_Name di dalam kurung
# scrape_all_imdb_reviews("Movie_ID", "Movie Name")

In [8]:
####  DreamWorks
# scrape_all_imdb_reviews("tt2850386", "The Croods: A New Age")
# scrape_all_imdb_reviews("tt3915174", "Puss in Boots: The Last Wish")
# scrape_all_imdb_reviews("tt6587640", "Trolls World Tour")
# scrape_all_imdb_reviews("tt6932874", "The Boss Baby: Family Business")
# scrape_all_imdb_reviews("tt8115900", "The Bad Guys")
# scrape_all_imdb_reviews("tt11084896", "Spirit Untamed" )
# scrape_all_imdb_reviews("tt14362112", "Trolls Band Together")
# scrape_all_imdb_reviews("tt21692408", "Kung Fu Panda 4")
# scrape_all_imdb_reviews("tt27155038", "Ruby Gillman: Teenage Kraken")
# scrape_all_imdb_reviews("tt28066777", "Orion and the Dark")
# scrape_all_imdb_reviews("tt29623480", "The Wild Robot")

In [9]:
#### Illumination
# scrape_all_imdb_reviews("tt6467266", "Sing 2")
# scrape_all_imdb_reviews("tt5113044", "Minions: The Rise of The Gru")
# scrape_all_imdb_reviews("tt6495056", "Migration")
# scrape_all_imdb_reviews("tt6718170", "The super mario bros the movie")
# scrape_all_imdb_reviews("tt7510222", "Despicable Me 4")

In [10]:
####  Pixar
# scrape_all_imdb_reviews("tt2948372", "Soul")
# scrape_all_imdb_reviews("tt7146812", "Onward")
# scrape_all_imdb_reviews("tt8097030", "Turning Red")
# scrape_all_imdb_reviews("tt10298810", "Lightyear")
# scrape_all_imdb_reviews("tt12801262", "Luca")
# scrape_all_imdb_reviews("tt15789038", "Elemental")
# scrape_all_imdb_reviews("tt22022452", "Inside Out 2")

In [11]:
#### Disney
# scrape_all_imdb_reviews("tt2953050", "Encanto")
# scrape_all_imdb_reviews("tt5109280", "Raya and the Last Dragon")
# scrape_all_imdb_reviews("tt10298840", "Strange World")
# scrape_all_imdb_reviews("tt11304740", "Wish")
# scrape_all_imdb_reviews("tt13622970", "Moana 2")

In [12]:
# VERSION 1
# !pip install selenium
# !apt-get update
# !apt install chromium-chromedriver

# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.keys import Keys
# from bs4 import BeautifulSoup
# import pandas as pd
# import time
# import os
# from google.colab import drive

# # Mount Google Drive
# drive.mount('/content/drive')

# # Install ChromeDriver in Google Colab
# def setup_selenium():
#     !apt-get update > /dev/null 2>&1
#     !apt install chromium-chromedriver > /dev/null 2>&1
#     os.environ["PATH"] += ":/usr/lib/chromium-browser/"

# setup_selenium()

# def scrape_all_imdb_reviews(movie_id, max_clicks=1000):
#     """Scrapes all IMDb reviews, ratings, and dates for a given movie ID using Selenium."""
#     url = f"https://www.imdb.com/title/{movie_id}/reviews/?ref_=tt_ururv_sm"
#     options = webdriver.ChromeOptions()
#     options.add_argument("--headless")
#     options.add_argument("--disable-gpu")
#     options.add_argument("--no-sandbox")
#     options.add_argument("--disable-dev-shm-usage")
#     options.add_argument("--remote-debugging-port=9222")  # Prevent session conflict in Colab
#     options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

#     driver = webdriver.Chrome(options=options)
#     driver.get(url)
#     time.sleep(5)  # Allow page to load

#     for _ in range(max_clicks):
#         try:
#             load_more_button = driver.find_element(By.XPATH, "//span[contains(@class, 'ipc-see-more__text')]")
#             driver.execute_script("arguments[0].click();", load_more_button)
#             time.sleep(2)
#         except Exception:
#             print("No more 'Load More' button found.")
#             break

#     # Click all 'Spoilers' buttons to reveal hidden reviews
#     while True:
#         try:
#             spoiler_buttons = driver.find_elements(By.XPATH, "//button[contains(@class, 'ipc-btn ipc-btn--single-padding ipc-btn--center-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-error ipc-text-button sc-3e6f8aa9-1 iXFqZy review-spoiler-button')]")
#             if not spoiler_buttons:
#                 break
#             for button in spoiler_buttons:
#                 driver.execute_script("arguments[0].click();", button)
#                 time.sleep(1)
#         except Exception:
#             print("No more spoilers buttons found.")
#             break

#     soup = BeautifulSoup(driver.page_source, "html.parser")
#     driver.quit()

#     reviews = []
#     review_containers = soup.find_all("article", class_="sc-8c92b587-1 cwztqu user-review-item")

#     for review in review_containers:
#         text = review.find("div", class_="ipc-html-content-inner-div")
#         rating = review.find("span", class_="ipc-rating-star ipc-rating-star--base ipc-rating-star--otherUserAlt review-rating")
#         date = review.find("div", class_="sc-f6867cfd-1 iHZNcU")
#         if date:
#             date = date.find("li", class_="ipc-inline-list__item review-date")

#         review_text = text.get_text(strip=True) if text else "No review text"
#         if review_text == "No review text":  # Fallback for hidden reviews
#             hidden_text = review.find("span", class_="sc-16ede01-2 gkqnAC")
#             review_text = hidden_text.get_text(strip=True) if hidden_text else "No review text"

#         review_rating = rating.find("span").get_text(strip=True) if rating else "No rating"
#         review_date = date.get_text(strip=True) if date else "No date"

#         reviews.append({
#             "Movie_ID": movie_id,
#             "Date": review_date,
#             "Review": review_text,
#             "Rating": review_rating,
#         })

#     print(f"Scraping completed. Total reviews collected: {len(reviews)}")
#     df = pd.DataFrame(reviews)
#     save_path = f"/content/drive/My Drive/Tugas Akhir/Dataset/{movie_id}_reviews.csv"
#     df.to_csv(save_path, index=False)
#     print(f"Reviews saved to {save_path}")
#     return df